# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
!git clone https://github.com/binitnayak/ml-internship-starter.git

fatal: destination path 'ml-internship-starter' already exists and is not an empty directory.


In [15]:
import os

path = "ml-internship-starter/data/raw/content_refresh_anonymized.csv"

print("Dataset exists:", os.path.exists(path))
print("Path:", path)

Dataset exists: True
Path: ml-internship-starter/data/raw/content_refresh_anonymized.csv


In [16]:
import pandas as pd
import numpy as np

df = pd.read_csv(path)

df["target_decline"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Dataset shape:", df.shape)
print(df["target_decline"].value_counts())

Dataset shape: (30000, 45)
target_decline
1    16262
0    13738
Name: count, dtype: int64


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



### Finding 1: AI Overviews and organic clicks

The FlyRank research reports that AI Overviews (AIOs) are associated with a reduction in organic clicks. The outcome being studied is organic click behavior when an AIO is present.

My methodology question is: how exactly was the click outcome labeled, and what was the comparison group? I would also want to know the sample size, time period, query mix, and geographic scope. If the validation was based on a limited sample or specific query types, the result should be described as a directional observation rather than a universal effect.

I see this as a useful finding, but the strength of the claim should match the validation design and the population that was actually measured.

### Finding 2: Content/GEO tactics and AI visibility

The research also reports that some content-format or GEO tactics can improve visibility in AI-generated results. The outcome is based on whether modified content becomes more visible or receives more AI citations.

My methodology question is: was this result measured using controlled experiments, real-world traffic, or both? If the result comes mainly from a controlled study, I would want to know whether it has been replicated across different AI systems, query types, and real-world websites.

The finding can provide a useful directional signal, but a controlled result should not automatically be generalized to every AI search system or every website.

### Overall methodology lesson

For both findings, I would separate the measured outcome from the broader interpretation. Before making a strong claim, I would check the label definition, sample, timeframe, comparison group, validation design, and whether the result was reproduced on independent data.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*



In Week-5/ML-08, I evaluated Logistic Regression using a stratified random row split. This provided a useful first benchmark, but pages from the same client could potentially appear in both training and test sets.

For this validation audit, I use a client-aware split. The same client is kept entirely in either the training set or the test set.

This is a stricter and more realistic test of whether the model can generalize beyond clients seen during training.

In [17]:
feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[feature_cols]
y = df["target_decline"]

groups = df["client_id"]

print("Number of features:", len(feature_cols))
print("Number of clients:", groups.nunique())

Number of features: 26
Number of clients: 32


In [18]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train_group))
print("Test rows:", len(X_test_group))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0


In [19]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Same model as ML-08
group_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train only on training clients
group_model.fit(
    X_train_group,
    y_train_group
)

# Predict unseen clients
y_group_pred = group_model.predict(X_test_group)

# Evaluate
group_accuracy = accuracy_score(
    y_test_group,
    y_group_pred
)

group_precision = precision_score(
    y_test_group,
    y_group_pred,
    zero_division=0
)

group_recall = recall_score(
    y_test_group,
    y_group_pred,
    zero_division=0
)

group_f1 = f1_score(
    y_test_group,
    y_group_pred,
    zero_division=0
)

print("Client-Aware Logistic Regression")
print("--------------------------------")
print(f"Accuracy : {group_accuracy:.4f}")
print(f"Precision: {group_precision:.4f}")
print(f"Recall   : {group_recall:.4f}")
print(f"F1 Score : {group_f1:.4f}")

Client-Aware Logistic Regression
--------------------------------
Accuracy : 0.7564
Precision: 0.8095
Recall   : 0.6843
F1 Score : 0.7417


### Before vs after interpretation

The original ML-08 random split produced an accuracy of 0.8225 and an F1 score of 0.8337.

Under the stricter client-aware split, accuracy decreased to 0.7564 and F1 decreased to 0.7417. Precision was 0.8095 and recall was 0.6843.

The performance decrease is important because the client-aware split tests the model on clients that were not present during training. This suggests that the random row split may have provided an optimistic estimate of generalization.

I therefore treat the client-aware results as a more cautious estimate of model performance on unseen clients. The model still provides a measured directional signal, but the results do not support a claim of reliable performance across all future clients or datasets.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [20]:
leakage_columns = [
    "target_decline",
    "trend_direction",
    "trend_pct",
    "client_id"
]

print("Leakage audit")
print("-------------")

for col in leakage_columns:
    if col in feature_cols:
        print(f"WARNING: {col} is included in model features")
    else:
        print(f"OK: {col} is not included in model features")

Leakage audit
-------------
OK: target_decline is not included in model features
OK: trend_direction is not included in model features
OK: trend_pct is not included in model features
OK: client_id is not included in model features


In [21]:
from sklearn.metrics import confusion_matrix

# Confusion matrix
cm = confusion_matrix(y_test_group, y_group_pred)

tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix")
print("----------------")
print(cm)

print("\nError Summary")
print("-------------")
print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

Confusion Matrix
----------------
[[2507  507]
 [ 994 2155]]

Error Summary
-------------
True Negatives : 2507
False Positives: 507
False Negatives: 994
True Positives : 2155


In [22]:
# Build error-analysis table
error_analysis = df.iloc[test_idx][[
    "content_id",
    "client_id",
    "target_decline"
]].copy()

error_analysis["prediction"] = y_group_pred

error_analysis["error_type"] = np.where(
    (error_analysis["target_decline"] == 1) &
    (error_analysis["prediction"] == 0),
    "False Negative",
    np.where(
        (error_analysis["target_decline"] == 0) &
        (error_analysis["prediction"] == 1),
        "False Positive",
        "Correct"
    )
)

errors = error_analysis[
    error_analysis["error_type"] != "Correct"
]

print("Total errors:", len(errors))

errors.head(10)

Total errors: 1501


,content_id,client_id,target_decline,prediction,error_type
13,content_a5a2fbc76336,client_8527a891e2,0,1,False Positive
23,content_2da6ae9d0882,client_e629fa6598,1,0,False Negative
25,content_033ae3e7aecf,client_f369cb89fc,1,0,False Negative
36,content_bce275871a25,client_f369cb89fc,0,1,False Positive
39,content_4595e8704e07,client_8527a891e2,1,0,False Negative
43,content_1938955b34c4,client_f369cb89fc,1,0,False Negative
47,content_40cb4af260c0,client_f369cb89fc,1,0,False Negative
49,content_f0717373e86e,client_8527a891e2,1,0,False Negative
51,content_d8a23b5e10c5,client_f369cb89fc,1,0,False Negative
54,content_ff8ea1364b59,client_e629fa6598,1,0,False Negative


In [23]:
print("False Negatives:")
display(
    errors[errors["error_type"] == "False Negative"].head(5)
)

print("\nFalse Positives:")
display(
    errors[errors["error_type"] == "False Positive"].head(5)
)

False Negatives:


,content_id,client_id,target_decline,prediction,error_type
23,content_2da6ae9d0882,client_e629fa6598,1,0,False Negative
25,content_033ae3e7aecf,client_f369cb89fc,1,0,False Negative
39,content_4595e8704e07,client_8527a891e2,1,0,False Negative
43,content_1938955b34c4,client_f369cb89fc,1,0,False Negative
47,content_40cb4af260c0,client_f369cb89fc,1,0,False Negative



False Positives:


,content_id,client_id,target_decline,prediction,error_type
13,content_a5a2fbc76336,client_8527a891e2,0,1,False Positive
36,content_bce275871a25,client_f369cb89fc,0,1,False Positive
64,content_685de0e3b7cb,client_f369cb89fc,0,1,False Positive
126,content_be5e23c0a35e,client_f369cb89fc,0,1,False Positive
179,content_552a9396d8dc,client_8527a891e2,0,1,False Positive


In [24]:
error_examples = df.iloc[test_idx][[
    "content_id",
    "target_decline"
]].copy()

error_examples["prediction"] = y_group_pred

error_examples["error_type"] = np.where(
    (error_examples["target_decline"] == 1) &
    (error_examples["prediction"] == 0),
    "False Negative",
    np.where(
        (error_examples["target_decline"] == 0) &
        (error_examples["prediction"] == 1),
        "False Positive",
        "Correct"
    )
)

error_examples = error_examples[
    error_examples["error_type"] != "Correct"
]

display(error_examples.head(10))

,content_id,target_decline,prediction,error_type
13,content_a5a2fbc76336,0,1,False Positive
23,content_2da6ae9d0882,1,0,False Negative
25,content_033ae3e7aecf,1,0,False Negative
36,content_bce275871a25,0,1,False Positive
39,content_4595e8704e07,1,0,False Negative
43,content_1938955b34c4,1,0,False Negative
47,content_40cb4af260c0,1,0,False Negative
49,content_f0717373e86e,1,0,False Negative
51,content_d8a23b5e10c5,1,0,False Negative
54,content_ff8ea1364b59,1,0,False Negative


### Error examples and interpretation

The client-aware test produced both false negatives and false positives.

The false-negative examples are pages that were observed as declining (`target_decline = 1`) but were predicted as non-declining (`prediction = 0`). The false-positive examples are pages that were not observed as declining (`target_decline = 0`) but were predicted as declining (`prediction = 1`).

These examples show that the model does not perfectly separate the two classes, even when evaluated on clients that were not used during training.

The errors should be interpreted as evidence of uncertainty rather than as evidence that the model is unusable. For a refresh-prioritization workflow, false negatives are particularly important because a declining page may be missed. False positives also matter because they may cause a page to receive review priority when it does not actually belong to the declining class.

The error examples support using the model as decision-support for human review rather than as an automatic refresh decision.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



### Original claim

"I build ML models that reliably predict which pages will decline in search performance."

### Revised claim

"I built and measured a Logistic Regression model that provides a directional signal for identifying pages associated with declining search performance. Under client-aware validation, the model can provide decision-support for prioritizing pages for further review and potential refresh."

### Why I changed the claim

The original claim uses the word "reliably," which is stronger than what the validation results establish.

The client-aware evaluation produced an F1 score of 0.7417, compared with 0.8337 under the original random split. This difference shows that performance is lower when the model is evaluated on unseen clients.

The revised claim therefore focuses on the observed and measured results and describes the model as directional decision-support rather than a guarantee of future search performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.